# Step 2: Text Simplification — T5-base (Custom Data)

This notebook fine-tunes **T5-base** using your completely custom, scraped parallel dataset (`scraped_simplification_pairs.csv`).

**Note for T5:**
T5 is a text-to-text model. It requires a task prefix to know what to do. We will use the prefix: `"simplify: "` before every input sentence.

In [ ]:
# ============================================================
#  INSTALL
# ============================================================
!pip install -q transformers pandas scikit-learn

# ============================================================
#  IMPORTS
# ============================================================
import os, torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import (
    T5ForConditionalGeneration,
    T5TokenizerFast,
    AdamW,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split

SAVE_DIR   = '/content/t5_simplifier_custom'
MODEL_NAME = 't5-base'
EPOCHS     = 3
BATCH_SIZE = 8
MAX_INPUT  = 128
MAX_TARGET = 128
LR         = 3e-4
PATIENCE   = 2

os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
#  LOAD CUSTOM SCRAPED DATASET
# ============================================================
print('\nLoading scraped_simplification_pairs.csv...')
try:
    df = pd.read_csv('/content/scraped_simplification_pairs.csv').dropna()
except FileNotFoundError:
    print("ERROR: Please upload scraped_simplification_pairs.csv generated in Step 1!")
    raise

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

train_pairs = train_df.to_dict('records')
val_pairs   = val_df.to_dict('records')

print(f'Train pairs : {len(train_pairs)}')
print(f'Val   pairs : {len(val_pairs)}')

# ============================================================
#  TOKENIZER + DATASET CLASS
# ============================================================
tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)

class SimplificationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        # T5 Prefix required
        src = "simplify: " + str(self.pairs[idx]['complex_text'])
        tgt = str(self.pairs[idx]['simple_text'])

        src_enc = tokenizer(
            src, max_length=MAX_INPUT, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        with tokenizer.as_target_tokenizer():
            tgt_enc = tokenizer(
                tgt, max_length=MAX_TARGET, padding='max_length',
                truncation=True, return_tensors='pt'
            )
        labels = tgt_enc['input_ids'].flatten()
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            'input_ids':      src_enc['input_ids'].flatten(),
            'attention_mask': src_enc['attention_mask'].flatten(),
            'labels':         labels
        }

train_loader = DataLoader(SimplificationDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(SimplificationDataset(val_pairs),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ============================================================
#  MODEL
# ============================================================
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
print(f'\nModel parameters: {sum(p.numel() for p in model.parameters()):,}')

# ============================================================
#  OPTIMIZER + SCHEDULER
# ============================================================
optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.05)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# ============================================================
#  TRAINING LOOP
# ============================================================
best_val_loss    = float('inf')
patience_counter = 0

print('\n' + '='*55)
print('       STARTING T5 SIMPLIFICATION TRAINING')
print('='*55)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            labels=batch['labels'].to(device)
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if step % 100 == 0 and step > 0:
            print(f'  Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                labels=batch['labels'].to(device)
            )
            val_loss += outputs.loss.item()
    avg_val_loss = val_loss / len(val_loader)

    print(f'\n--- Epoch {epoch+1}/{EPOCHS} ---')
    print(f'  Train Loss : {avg_train_loss:.4f}')
    print(f'  Val   Loss : {avg_val_loss:.4f}')

    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        patience_counter = 0
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f'  --> New best! Model saved to {SAVE_DIR}\n')
    else:
        patience_counter += 1
        print(f'  --> No improvement. Patience: {patience_counter}/{PATIENCE}\n')
        if patience_counter >= PATIENCE:
            print('  Early stopping triggered.')
            break

# ============================================================
#  INFERENCE DEMO
# ============================================================
print('\n' + '='*55)
print('         INFERENCE DEMO (BEST MODEL)')
print('='*55)

model = T5ForConditionalGeneration.from_pretrained(SAVE_DIR).to(device)
model.eval()

test_sentences = val_df['complex_text'].head(3).tolist()

for i, sent in enumerate(test_sentences):
    inp_text = "simplify: " + sent
    inp = tokenizer(inp_text, return_tensors='pt', max_length=MAX_INPUT, truncation=True).to(device)
    out = model.generate(**inp, max_length=80, num_beams=4, early_stopping=True)
    simplified = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f'\n[{i+1}] Original  : {sent}')
    print(f'    Simplified: {simplified}')

# ============================================================
#  ZIP AND DOWNLOAD
# ============================================================
import shutil
shutil.make_archive('/content/t5_simplifier_custom', 'zip', SAVE_DIR)
from google.colab import files
files.download('/content/t5_simplifier_custom.zip')
print('\nDownloaded t5_simplifier_custom.zip')